# RQ3 — Structural Side Effects of LLM-Based Refactoring
### Google Colab Analysis Notebook

> **Research Question:** What unintended structural side effects emerge from LLM-based refactoring?

| Sub-Q | Focus |
|-------|-------|
| **RQ3a** | Smell side effects — does refactoring remove the target smell without introducing new ones? |
| **RQ3b** | Coverage side effects — does refactoring preserve, improve, or degrade test coverage? |

---

### ▶ How to use this notebook

1. Run **cell 1** to install dependencies  
2. Run **cell 2** to import all libraries  
3. Run **cell 3** — upload `rq3a_raw_smells.csv` when prompted  
4. Run **cell 4** — upload `rq3b_raw_coverage.csv` when prompted  
5. Run all remaining cells in order to generate charts and tables

> **Where do the CSVs come from?**  
> Export them from the research dashboard: **↓ Raw Smells CSV** and **↓ Raw Coverage CSV** buttons on the RQ3 page.


## 1 — Install Dependencies

In [7]:
# Install required libraries (seaborn not pre-installed on all Colab runtimes)
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

try:
    import seaborn
except ImportError:
    _install('seaborn')
    print('seaborn installed ✓')

print('All dependencies ready ✓')


→ Path to the SQLite research database that will be used for all queries:
  DB path: /home/gabriel/Disk/Research/research-javascript-test-smells/research_data/research.db
  DB exists: True  (must be True before continuing)


In [8]:
## 2 — Imports & Plotting Config
import os
import json
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
import seaborn as sns
warnings.filterwarnings('ignore')

# ── Purple theme (consistent with RQ1 blue / RQ2 green) ──────────────────────
PURPLE    = '#7c3aed'
PURPLE_L  = '#a78bfa'
PURPLE_LL = '#ede9fe'
RED       = '#ef4444'
GREEN     = '#22c55e'
AMBER     = '#f59e0b'
GRAY      = '#9ca3af'

plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'axes.edgecolor':    '#e5e7eb',
    'axes.grid':         True,
    'grid.color':        '#f3f4f6',
    'grid.linestyle':    '-',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'font.family':       'DejaVu Sans',
    'axes.titlesize':    13,
    'axes.labelsize':    11,
})

MODEL_PALETTE = [
    '#7c3aed', '#a855f7', '#6d28d9', '#9333ea',
    '#4f46e5', '#7e22ce', '#c026d3', '#db2777',
]

print('→ All libraries imported and purple theme configured ✓')


ModuleNotFoundError: No module named 'matplotlib'

## 3 — Load Input CSVs

Upload the two CSV files exported from the research dashboard:

| File | Button on the dashboard |
|------|------------------------|
| `rq3a_raw_smells.csv` | **↓ Raw Smells CSV** |
| `rq3b_raw_coverage.csv` | **↓ Raw Coverage CSV** |

In [ ]:
# ── Cell 3: Upload rq3a_raw_smells.csv ───────────────────────────────────────
# This CSV contains one row per experiment with after-phase smell data.
# Columns include: experiment_id, smell_type, model, prompt, failure_type,
#   is_error_class, smell_removed, introduced_new_smells, added_smells_json,
#   total_added
#
# Source: "↓ Raw Smells CSV" button on the RQ3 dashboard page.

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import files as _colab_files
    print('📂 Please upload rq3a_raw_smells.csv ...')
    uploaded = _colab_files.upload()
    _fname = next(
        (k for k in uploaded if 'smells' in k.lower() and 'raw' in k.lower()),
        list(uploaded.keys())[0]
    )
    import io
    df_a = pd.read_csv(io.BytesIO(uploaded[_fname]))
else:
    # Local / VS Code: edit this path if needed
    _fname = 'rq3a_raw_smells.csv'
    df_a = pd.read_csv(_fname)

# ── Ensure total_added column ─────────────────────────────────────────────────
def _total_added(js):
    if not js or str(js) in ('{}', 'nan'):
        return 0
    try:
        return sum(json.loads(js).values())
    except Exception:
        return 0

if 'total_added' not in df_a.columns:
    df_a['total_added'] = df_a.get('added_smells_json', df_a.get('added_smells', '{}')).apply(_total_added)

# Normalise column name (dashboard may export as 'added_smells' instead of 'added_smells_json')
if 'added_smells_json' not in df_a.columns and 'added_smells' in df_a.columns:
    df_a['added_smells_json'] = df_a['added_smells']

print('→ RQ3a dataset loaded from CSV. Each row is one experiment where after-phase')
print('  smell detection ran successfully. added_smells_json is a dict {smell_type: count_added}')
print('  for categories where count_after > count_before. total_added = sum of all counts.')
print(f'  Rows: {len(df_a):,}  |  Columns: {list(df_a.columns)}')
df_a.head(3)


In [ ]:
# ── Cell 4: Upload rq3b_raw_coverage.csv ─────────────────────────────────────
# This CSV contains one row per experiment with before/after coverage data.
# Columns include: experiment_id, smell_type, model, prompt, failure_type,
#   before_stmt, after_stmt, delta_stmt, delta_branch, delta_fn, delta_lines,
#   coverage_decreased
#
# Source: "↓ Raw Coverage CSV" button on the RQ3 dashboard page.

if IN_COLAB:
    print('📂 Please upload rq3b_raw_coverage.csv ...')
    uploaded = _colab_files.upload()
    _fname = next(
        (k for k in uploaded if 'coverage' in k.lower()),
        list(uploaded.keys())[0]
    )
    df_b = pd.read_csv(io.BytesIO(uploaded[_fname]))
else:
    _fname = 'rq3b_raw_coverage.csv'
    df_b = pd.read_csv(_fname)

# Ensure delta columns exist (dashboard may name them differently)
_col_map = {
    'delta_statements': 'delta_stmt',
    'delta_branches':   'delta_branch',
    'delta_functions':  'delta_fn',
}
df_b.rename(columns=_col_map, inplace=True)

print('→ RQ3b dataset loaded from CSV. Each row is one experiment where the pipeline')
print('  collected coverage data in both phases (requires --coverage flag). Delta columns')
print('  measure the percentage-point change after refactoring (positive = improved).')
print(f'  Rows: {len(df_b):,}  |  Columns: {list(df_b.columns)}')
df_b.head(3)


In [ ]:
# ── Data availability overview ────────────────────────────────────────────────
# In Colab mode all counts are derived from the loaded CSVs; no DB access needed.
total_exp   = len(df_a)          # experiments with smell data available
# pending_csv is not recoverable from the exported CSV (those rows were excluded)
# We report a note instead.
pending_csv_note = 'N/A (excluded from CSV export)'

avail = {
    'Total experiments (in CSV)':  total_exp,
    'RQ3a — included':             len(df_a),
    'RQ3a — error class':          df_a['is_error_class'].sum(),
    'RQ3a — no after CSV (NULL)':  pending_csv_note,
    'RQ3b — included':             len(df_b),
}

print('→ Data availability breakdown — shows how many experiments are usable for each sub-question.')
print('  "Included" for RQ3a = all rows in the uploaded rq3a_raw_smells.csv.')
print('  "Error class" = pipeline failed with a runner error; counted separately.')
print('  "No after CSV" is not available in the exported CSV (those rows were excluded upstream).')
print()
print('=' * 55)
print('DATA AVAILABILITY')
print('=' * 55)
for k, v in avail.items():
    if isinstance(v, int):
        print(f'  {k:<40} {int(v):>8,}')
    else:
        print(f'  {k:<40} {str(v):>8}')
print()

WARN_3A = len(df_a) < 100
WARN_3B = len(df_b) < 100
if WARN_3A: print('⚠️  WARNING: RQ3a has fewer than 100 experiments — results may not be representative.')
if WARN_3B: print('⚠️  WARNING: RQ3b has fewer than 100 experiments — ensure pipeline ran with --coverage.')


## 3  RQ3a — Smell Side Effects

### 3.1  Overall Summary

In [ ]:
n_a = len(df_a)
total_added_instances = int(df_a['total_added'].sum())

overall_a = {
    'n':                           n_a,
    'removed':                     int(df_a['smell_removed'].sum()),
    'removal_rate_%':              round(df_a['smell_removed'].mean() * 100, 1) if n_a else None,
    'new_introduced_exps':         int(df_a['introduced_new_smells'].sum()),
    'new_introduction_rate_%':     round(df_a['introduced_new_smells'].mean() * 100, 1) if n_a else None,
    'total_added_smell_instances': total_added_instances,
    'avg_added_per_experiment':    round(df_a['total_added'].mean(), 3) if n_a else None,
}
print('→ RQ3a aggregated summary across ALL included experiments (regardless of smell type or model).')
print('  removal_rate_%          — share of experiments where the target smell was successfully removed.')
print('  new_introduction_rate_% — share of experiments where ≥1 new (unrelated) smell was added.')
print('  total_added_smell_instances — sum of all added counts from added_smells JSON across all experiments.')
print('  avg_added_per_experiment — mean total per-experiment addition count (0 = no side effects on average).')
print()
print('RQ3a Overall:')
for k, v in overall_a.items():
    print(f'  {k:<40} {v}')


In [ ]:
# ── T1: By smell type ─────────────────────────────────────────────────────────
t1 = (
    df_a.groupby('smell_type', dropna=False)
    .agg(
        n=('experiment_id', 'count'),
        removed=('smell_removed', 'sum'),
        new_introduced=('introduced_new_smells', 'sum'),
        avg_added=('total_added', 'mean'),
    )
    .reset_index()
)
t1['removal_rate_%']          = (t1['removed']       / t1['n'] * 100).round(1)
t1['new_introduction_rate_%'] = (t1['new_introduced'] / t1['n'] * 100).round(1)
t1['avg_added']               = t1['avg_added'].round(3)
t1 = t1.sort_values('removal_rate_%', ascending=False)
t1.to_csv('rq3a_by_smell_type.csv', index=False)
print('→ T1 saved. This table breaks down smell removal and new-smell introduction by smell type,')
print('  sorted by removal rate (highest first). avg_added is the mean total count of smell')
print('  instances added per experiment for that smell type (from added_smells JSON).')
print('  rq3a_by_smell_type.csv')
t1


In [ ]:
# ── G1: Removal rate vs new introduction rate by smell type ──────────────────
fig, ax = plt.subplots(figsize=(max(8, len(t1) * 0.9), 5))

x     = np.arange(len(t1))
width = 0.38
labels = t1['smell_type'].fillna('Unknown').tolist()

bars1 = ax.bar(x - width/2, t1['removal_rate_%'],          width, label='Removal Rate %',            color=PURPLE,   alpha=0.92, zorder=3)
bars2 = ax.bar(x + width/2, t1['new_introduction_rate_%'], width, label='New Introduction Rate %',   color=AMBER,    alpha=0.92, zorder=3)

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 1, f'{h:.0f}%',
                ha='center', va='bottom', fontsize=8, color='#374151')

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('% of included experiments')
ax.set_ylim(0, 115)
ax.set_title('G1 — Smell Removal vs New Introduction Rate by Smell Type', fontweight='bold')
ax.legend(frameon=True, framealpha=0.9, fontsize=9)
plt.tight_layout()
plt.savefig('rq3a_g1_removal_vs_newintro.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ G1 chart saved. Purple bars show the removal rate per smell type; amber bars show the rate')
print('  at which each smell type accidentally triggered new smell introductions. Ideally purple')
print('  bars should be tall (high removal) and amber bars short (low side-effect rate).')
print('  rq3a_g1_removal_vs_newintro.png')

In [ ]:
# ── T2: New smell taxonomy — parsed from added_smells_json ───────────────────
# Parse the JSON dict from every experiment that has at least one added smell,
# and aggregate counts by smell category.
from collections import Counter

tax: Counter = Counter()
for js in df_a[df_a['introduced_new_smells'] == 1]['added_smells_json']:
    if js and js != '{}':
        try:
            for stype, cnt in json.loads(js).items():
                tax[stype] += cnt
        except Exception:
            pass

t2 = pd.DataFrame([
    {'introduced_smell_type': k, 'count': v}
    for k, v in tax.most_common()
])
if t2.empty:
    print('No added smells found.')
else:
    total_new = t2['count'].sum() or 1
    n_incl    = len(df_a) or 1
    t2['pct_of_new']      = (t2['count'] / total_new * 100).round(1)
    t2['pct_of_included'] = (t2['count'] / n_incl  * 100).round(1)
    t2.to_csv('rq3a_new_smell_taxonomy.csv', index=False)
    print('→ T2 saved. Taxonomy of added smell categories, derived from experiments.added_smells JSON.')
    print('  Each entry: a smell type whose count increased after refactoring.')
    print('  "pct_of_new" = share of all added instances; "pct_of_included" = prevalence across')
    print(f'  all {n_incl:,} included experiments.')
    print(f'  T2 → rq3a_new_smell_taxonomy.csv  (total added instances: {total_new:,})')
    print(t2.to_string(index=False))


In [ ]:
# ── G5: Smell interaction matrix (targeted smell → accidentally added smells) ─
# Rows = targeted smell type; columns = accidentally introduced smell type.
# Cell value = total added instances across all experiments.
# Uses the `tax` Counter built in the T2 cell above for column ordering.
from collections import defaultdict as _dd
import numpy as np
try:
    import seaborn as sns
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'seaborn', '-q'])
    import seaborn as sns

# Build interaction dict: {targeted_smell: Counter({added_smell: total_count})}
interact_raw = _dd(Counter)
for _, row in df_a[df_a['added_smells_json'].notna()].iterrows():
    js  = row['added_smells_json']
    tgt = row.get('smell_type') or 'Unknown'
    if js and js != '{}':
        try:
            for stype, cnt in json.loads(js).items():
                interact_raw[tgt][stype] += cnt
        except Exception:
            pass

if not interact_raw:
    print('No interaction data available (no experiments with added smells).')
else:
    row_labels = sorted(interact_raw.keys())
    col_labels = [s for s, _ in tax.most_common()]   # T2 Counter — ordered by frequency

    matrix = np.array(
        [[interact_raw[t].get(a, 0) for a in col_labels] for t in row_labels],
        dtype=float
    )

    # Annotation: show count only for non-zero cells
    annot = np.where(matrix > 0, matrix.astype(int).astype(str), '')

    fig, ax = plt.subplots(
        figsize=(max(8, len(col_labels) * 1.05 + 2.5),
                 max(4, len(row_labels) * 0.65 + 2.0))
    )
    sns.heatmap(
        matrix,
        xticklabels=col_labels,
        yticklabels=row_labels,
        annot=annot,
        fmt='',
        cmap='Purples',
        linewidths=0.5,
        linecolor='#ede9fe',
        cbar_kws={'label': 'Added instances', 'shrink': 0.75},
        ax=ax,
    )
    ax.set_xlabel('Accidentally added smell type', fontsize=10, labelpad=10)
    ax.set_ylabel('Targeted smell type (being refactored)', fontsize=10, labelpad=10)
    ax.set_title(
        'G5 — Smell Interaction Matrix\n'
        'How often does refactoring smell X accidentally introduce smell Y?',
        fontweight='bold', pad=14
    )
    plt.xticks(rotation=30, ha='right', fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()
    plt.savefig('rq3a_g5_interaction_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()

    total_pairs = int((matrix > 0).sum())
    top_cell    = max(((int(matrix[i,j]), row_labels[i], col_labels[j])
                       for i in range(len(row_labels))
                       for j in range(len(col_labels))
                       if matrix[i,j] > 0), default=None)

    print('→ G5 saved.  rq3a_g5_interaction_matrix.png')
    print('  Each row is the smell type being refactored; each column is a smell type')
    print('  accidentally introduced. Darker cells = more added instances.')
    print(f'  Non-zero pairs: {total_pairs}  |  Distinct targeted smells: {len(row_labels)}')
    if top_cell:
        c, tgt, add = top_cell
        print(f'  Strongest interaction: refactoring "{tgt}" → adds "{add}" ({c} instances)')


In [ ]:
# ── G2: New smell introduction rate by model ─────────────────────────────────
by_model_a = (
    df_a.groupby('model', dropna=False)
    .agg(n=('experiment_id','count'), new_introduced=('introduced_new_smells','sum'))
    .reset_index()
)
by_model_a['new_introduction_rate_%'] = (by_model_a['new_introduced'] / by_model_a['n'] * 100).round(1)
by_model_a = by_model_a.sort_values('new_introduction_rate_%', ascending=True)

colors = [MODEL_PALETTE[i % len(MODEL_PALETTE)] for i in range(len(by_model_a))]
fig, ax = plt.subplots(figsize=(max(6, len(by_model_a) * 1.1), 5))
bars = ax.bar(
    by_model_a['model'].fillna('Unknown'),
    by_model_a['new_introduction_rate_%'],
    color=colors, alpha=0.92, zorder=3
)
for bar in bars:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.8, f'{h:.1f}%',
                ha='center', va='bottom', fontsize=9, color='#374151')
ax.set_ylabel('New smell introduction rate (%)')
ax.set_xlabel('Model')
ax.set_ylim(0, min(110, by_model_a['new_introduction_rate_%'].max() * 1.25 + 10))
ax.set_title('G2 — New Smell Introduction Rate by Model', fontweight='bold')
plt.xticks(rotation=20, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig('rq3a_g2_new_smell_by_model.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ G2 chart saved. Each bar shows the percentage of experiments (for that model) where')
print('  at least one new, previously absent smell was introduced after refactoring. Models')
print('  are sorted from lowest to highest side-effect rate — a shorter bar is better.')
print('  rq3a_g2_new_smell_by_model.png')

## 4  RQ3b — Coverage Side Effects

### 4.1  Overall Summary

In [ ]:
n_b = len(df_b)

# Coverage classification per experiment
df_b['classification'] = np.where(
    (df_b[['delta_stmt','delta_branch','delta_fn','delta_lines']] > 0).any(axis=1), 'improved',
    np.where(
        (df_b[['delta_stmt','delta_branch','delta_fn','delta_lines']] < 0).any(axis=1), 'degraded',
        'preserved'
    )
)

counts = df_b['classification'].value_counts()
print('→ RQ3b aggregated summary across ALL included experiments (experiments with coverage data).')
print('  avg delta_* — mean percentage-point change per coverage metric after refactoring.')
print('    Positive = coverage went UP on average; negative = it went DOWN.')
print('  Classification rules: "improved" if any metric increased, "degraded" if any decreased')
print('  (and none increased), "preserved" if all four metrics stayed exactly the same.')
print()
print('RQ3b Overall:')
print(f'  n = {n_b:,}')
for metric in ['delta_stmt', 'delta_branch', 'delta_fn', 'delta_lines']:
    print(f'  avg {metric:<20} = {df_b[metric].mean():+.4f}')
print()
print('Coverage classification:')
for cls in ['improved', 'preserved', 'degraded']:
    c = counts.get(cls, 0)
    print(f'  {cls:<12} {c:>5,}  ({c/n_b*100:.1f}%)' if n_b else f'  {cls:<12}   0')

In [ ]:
# ── G3: Mean Δ statement coverage by model ────────────────────────────────────
by_model_b = (
    df_b.groupby('model', dropna=False)
    .agg(
        n=('experiment_id','count'),
        avg_delta_stmt=('delta_stmt','mean'),
        degraded=('coverage_decreased','sum'),
    )
    .reset_index()
)
by_model_b['degraded_rate_%'] = (by_model_b['degraded'] / by_model_b['n'] * 100).round(1)
by_model_b['avg_delta_stmt']  = by_model_b['avg_delta_stmt'].round(4)
by_model_b = by_model_b.sort_values('avg_delta_stmt', ascending=False)

bar_colors = [GREEN if v >= 0 else RED for v in by_model_b['avg_delta_stmt']]
fig, ax = plt.subplots(figsize=(max(6, len(by_model_b) * 1.1), 5))
bars = ax.bar(
    by_model_b['model'].fillna('Unknown'),
    by_model_b['avg_delta_stmt'],
    color=bar_colors, alpha=0.9, zorder=3
)
for bar in bars:
    h = bar.get_height()
    va = 'bottom' if h >= 0 else 'top'
    offset = 0.001 if h >= 0 else -0.001
    ax.text(bar.get_x() + bar.get_width()/2, h + offset, f'{h:+.3f}',
            ha='center', va=va, fontsize=8.5, color='#374151')
ax.axhline(0, color='#374151', linewidth=1.5, zorder=4)
ax.set_ylabel('Mean Δ Statement Coverage')
ax.set_xlabel('Model')
ax.set_title('G3 — Mean Δ Statement Coverage by Model', fontweight='bold')
plt.xticks(rotation=20, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig('rq3b_g3_coverage_delta_by_model.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ G3 chart saved. Each bar shows the mean change in statement coverage (after minus before)')
print('  for all experiments run with that model. Green bars = the model tended to preserve or')
print('  improve coverage on average; red bars = the model tended to reduce it. The zero baseline')
print('  is drawn as a reference line — bars crossing it are particularly noteworthy.')
print('  rq3b_g3_coverage_delta_by_model.png')

In [ ]:
# ── G4: Coverage classification stacked bar ───────────────────────────────────
counts_b = df_b['classification'].value_counts()
pcts = {cls: counts_b.get(cls, 0) / max(n_b, 1) * 100 for cls in ['improved', 'preserved', 'degraded']}

fig, ax = plt.subplots(figsize=(7, 2.8))
left = 0
for cls, color, label in [
    ('improved',  GREEN,    f"Improved ({pcts['improved']:.1f}%)"),
    ('preserved', PURPLE_L, f"Preserved ({pcts['preserved']:.1f}%)"),
    ('degraded',  RED,      f"Degraded ({pcts['degraded']:.1f}%)"),
]:
    v = pcts[cls]
    ax.barh('All experiments', v, left=left, color=color, alpha=0.9, height=0.55, label=label)
    if v > 3:
        ax.text(left + v/2, 0, f'{v:.1f}%', ha='center', va='center',
                fontsize=10, fontweight='bold', color='white')
    left += v
ax.set_xlim(0, 100)
ax.set_xlabel('% of included experiments')
ax.set_title('G4 — Coverage Classification (overall)', fontweight='bold')
ax.legend(loc='lower right', frameon=True, framealpha=0.9, fontsize=9)
ax.set_yticks([])
ax.spines['left'].set_visible(False)
plt.tight_layout()
plt.savefig('rq3b_g4_coverage_classification.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ G4 chart saved. This single stacked bar gives an at-a-glance view of the overall coverage')
print('  impact of refactoring: green = at least one coverage metric improved; purple = all metrics')
print('  stayed exactly the same; red = at least one metric degraded (and none improved).')
print('  rq3b_g4_coverage_classification.png')

In [ ]:
# ── T3: Coverage delta by smell type ─────────────────────────────────────────
t3 = (
    df_b.groupby('smell_type', dropna=False)
    .agg(
        n=('experiment_id','count'),
        avg_delta_stmt=('delta_stmt','mean'),
        avg_delta_branch=('delta_branch','mean'),
        avg_delta_fn=('delta_fn','mean'),
        avg_delta_lines=('delta_lines','mean'),
        degraded=('coverage_decreased','sum'),
    )
    .reset_index()
)
t3['degraded_rate_%'] = (t3['degraded'] / t3['n'] * 100).round(1)
for col in ['avg_delta_stmt','avg_delta_branch','avg_delta_fn','avg_delta_lines']:
    t3[col] = t3[col].round(4)
t3 = t3.sort_values('avg_delta_stmt', ascending=False)
t3.to_csv('rq3b_by_smell_type.csv', index=False)
print('→ T3 saved. This table shows coverage impact broken down by smell type — useful for')
print('  identifying whether certain smells are harder to refactor without hurting coverage.')
print('  Sorted by avg_delta_stmt (highest = coverage best preserved or most improved).')
print('  rq3b_by_smell_type.csv')
t3

# ── T4: Coverage delta by model ───────────────────────────────────────────────
print()
print('→ T4: Coverage delta aggregated by model. Shows which models are most likely to degrade')
print('  statement coverage (degraded_rate_%) and the mean coverage change they produce.')
t4 = by_model_b.copy()
t4.to_csv('rq3b_by_model.csv', index=False)
print('  rq3b_by_model.csv')
t4